# 05 · Estadísticas descriptivas  *(sección 5 del script)*

**Qué hacemos:**

1. Contamos cuántos pedidos hay por estado (`order_status`) y por medio de pago principal, mostrando no solo la cantidad sino **qué % del total** representa cada categoría.
2. Calculamos estadísticos (media, mediana, desvío, mín/máx, percentiles) de las variables numéricas clave, **incluyendo las nuevas** (`distancia_km`, `volumen_cm3`, `items_por_pedido`).

**Para qué:** tener una **foto rápida de la composición del dataset** antes de graficar nada, y detectar de entrada si alguna categoría es tan chica que no vale la pena analizarla por separado. Es el "reconocimiento previo" del terreno: sin estos números, los gráficos del notebook 06 no se pueden interpretar.

## Celda estándar: carga del artefacto

**Qué hacemos:** cargamos `02_df_features.csv` (notebook 04) y rehidratamos los tipos de fecha. Esta misma celda se repite en los notebooks 06 → 10 (está explicada celda por celda en `01_configuracion_inicial.ipynb`).

**Para qué:** partir de la tabla "lista para analizar" sin repetir limpieza ni features. Este notebook en particular solo imprime tablas (no grafica).

In [1]:
# Celda estándar (detallada en 01_configuracion_inicial.ipynb)
%matplotlib inline

import re
import unicodedata
import warnings
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")

BASE = Path.cwd()
if not (BASE / "data").exists() and (BASE.parent / "data").exists():
    BASE = BASE.parent

CSV_UNIFICADO = BASE / "data" / "olist_dataset_unificado.csv"
PIPELINE_DIR = BASE / "data" / "pipeline"
FIGS_DIR = BASE / "figuras_eda"

OLIST_BLUE = "#0A4EE4"
OLIST_BLUE_DARK = "#0D366B"
OLIST_GRAY = "#52514E"
PALETA_CATEGORICA = ["#0A4EE4", "#eb6834", "#1baf7a", "#eda100",
                     "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
SUDESTE = {"SP", "RJ", "MG", "ES"}

sns.set_theme(style="whitegrid", rc={
    "axes.edgecolor": "#c3c2b7",
    "axes.labelcolor": OLIST_GRAY,
    "text.color": "#0b0b0b",
    "xtick.color": OLIST_GRAY,
    "ytick.color": OLIST_GRAY,
    "font.family": "sans-serif",
})
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.titleweight"] = "bold"
pd.set_option("display.max_columns", 50)

# --- Carga del artefacto del notebook 04 ---
COLUMNAS_FECHA = [
    "shipping_limit_date", "order_purchase_timestamp", "order_approved_at",
    "order_delivered_carrier_date", "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

RUTA_FEATURES = PIPELINE_DIR / "02_df_features.csv"
if not RUTA_FEATURES.exists():
    raise FileNotFoundError(
        f"No existe {RUTA_FEATURES}. Ejecutá primero los notebooks 02, 03 y 04."
    )

df = pd.read_csv(RUTA_FEATURES)
for col in COLUMNAS_FECHA:
    df[col] = pd.to_datetime(df[col], errors="coerce")
df["mes_compra"] = df["order_purchase_timestamp"].dt.to_period("M").dt.to_timestamp()

print(f"filas={df.shape[0]:,} columnas={df.shape[1]}")

filas=112,650 columnas=43


## Conteos con porcentaje

**Qué hacemos:** armamos la función `resumen_conteo` que devuelve una tabla con el **conteo** (`value_counts`) y el **% del total** (`value_counts(normalize=True)`) de cualquier columna categórica, y la aplicamos a `order_status` y `payment_type_principal`.

**Para qué:** un conteo solo no alcanza: saber que hay X pedidos "shipped" no dice mucho si no sabés qué % del total representan. El % permite decidir al toque si una categoría es relevante o es ruido.

In [2]:
def resumen_conteo(serie, nombre="valor"):
    conteo = serie.value_counts()
    porcentaje = (serie.value_counts(normalize=True) * 100).round(1)
    return pd.DataFrame({nombre: conteo, "%": porcentaje})


print("--- order_status ---")
print(resumen_conteo(df["order_status"], "pedidos"))
print()
print("--- payment_type_principal ---")
print(resumen_conteo(df["payment_type_principal"], "pedidos"))

--- order_status ---
              pedidos     %
order_status               
delivered      110197  97.8
shipped          1185   1.1
canceled          542   0.5
invoiced          359   0.3
processing        357   0.3
unavailable         7   0.0
approved            3   0.0

--- payment_type_principal ---
                        pedidos     %
payment_type_principal               
credit_card               85063  75.5
boleto                    22867  20.3
voucher                    3028   2.7
debit_card                 1689   1.5


**Insight:** la enorme mayoría de los pedidos está en estado **"delivered" (entregado)** — el resto (canceled, shipped, unavailable…) son minoritarios, lo que explica por qué hay tan pocos NaN en las variables de tiempo de entrega en proporción al total. Y en pagos, **la tarjeta de crédito domina** con creces (crédito + boleto/boucher cubren casi todo).

## Estadísticos de las variables numéricas clave

**Qué hacemos:** `describe()` sobre 10 variables: precio, flete, `flete_ratio`, tiempos de entrega, puntaje de reseña, distancia, volumen, peso e ítems por pedido.

**Para qué:** para tener de un vistazo **dónde está el centro** (media/mediana), **cuánto varían** (desvío) y **qué tan colgadas están las colas** (mín/máx/percentiles) las variables con las que se va a trabajar.

> **Cómo leerlo:** si la **media > mediana**, la distribución tiene **cola a la derecha** (unos pocos valores muy grandes tiran de la media hacia arriba) — es lo habitual en precio, flete y distancia.

In [3]:
df[["price", "freight_value", "flete_ratio", "payment_value_total",
    "tiempo_entrega_dias", "dias_vs_estimado", "review_score",
    "distancia_km", "volumen_cm3", "items_por_pedido"]].describe().round(2)

,price,freight_value,flete_ratio,payment_value_total,tiempo_entrega_dias,dias_vs_estimado,review_score,distancia_km,volumen_cm3,items_por_pedido
count,112650.00,112650.00,112650.00,112647.00,110196.00,110196.00,111708.00,112087.00,112632.00,112650.00
mean,120.65,19.99,0.32,180.28,12.01,-12.03,4.03,596.66,15243.71,1.40
std,183.63,15.81,0.35,272.85,9.45,10.16,1.39,588.79,23418.52,1.12
min,0.85,0.00,0.00,9.59,0.00,-147.00,1.00,0.00,168.00,1.00
25%,39.90,13.08,0.13,65.67,6.00,-17.00,4.00,183.97,2851.50,1.00
50%,74.99,16.26,0.23,114.44,10.00,-13.00,5.00,431.61,6480.00,1.00
75%,134.90,21.15,0.39,195.39,15.00,-7.00,5.00,792.19,18375.00,1.00
max,6735.00,409.68,26.24,13664.08,209.00,188.00,5.00,3927.41,296208.00,21.00


**Insight:**

- **`flete_ratio`:** el flete promedio representa una porción considerable del precio del producto — consistente con un marketplace que envía a todo Brasil; en categorías de precio bajo, el envío puede pesar proporcionalmente mucho más.
- **`items_por_pedido`:** la mediana es 1 → la mayoría de los pedidos trae un solo ítem; los pedidos multi-ítem son la excepción, no la regla (dato que retoma el notebook 06).
- **`dias_vs_estimado`:** la media es negativa (se entrega antes de lo estimado en promedio), pero el percentil 75 es positivo → hay una cola de pedidos que sí se pasan.
- **`distancia_km`:** media > mediana → cola a la derecha: hay envíos larguísimos (Brasil es enorme) que promuean la distancia media.

**Siguiente paso:** `06_visualizaciones_exploratorias.ipynb` — llevar todos estos números a gráficos.